In [2]:
"""
avenue_31_v8_mapped_isotropic.py
(1) Ray equation in log-gradient form: dv/ds = w - (v.w)v, w = grad(ln n)
    -> removes n-division/clip bias; target k_cal ~ 2.60.
(2) Adds the CORRECTLY mapped GR isotropic completion:
    x(u) = (2 - u - 2*sqrt(1-u))/u ;  n = (1+x)^3/(1-x)  (pole at u=1).
(3) Re-runs the calibrated fork: single / double / mis-mapped / mapped / GR-control.
"""
import numpy as np

MASTER, R_CORE, N_R = 44.0, 20.0, 220

def solve_LM(act=0.055):
    r = np.arange(1, N_R + 1, dtype=float)
    rho = np.where(r <= R_CORE, (29.0 / MASTER) * act, 0.0)
    M_enc = np.cumsum(rho * 4 * np.pi * r ** 2)
    L = np.zeros_like(r); L[-1] = M_enc[-1] / r[-1]
    for i in range(N_R - 2, -1, -1):
        L[i] = L[i + 1] + M_enc[i] / r[i] ** 2
    return r, L, M_enc[-1]

r, L, M_eff = solve_LM()
u = np.clip(2 * L / MASTER, 0, 1 - 1e-9)
Rs = r[np.where(2 * L >= MASTER)[0][-1]]

x = (2 - u - 2 * np.sqrt(1 - u)) / np.maximum(u, 1e-12)
COMPLETIONS = {
    'single (u=1)':      1.0 / (1 - u),
    'double (u=1,2)':    1.0 / (np.sqrt(1 - u) * (1 - u / 2)),
    'iso mis-mapped':    (1 + u / 4) ** 3 / (1 - u / 4),
    'iso MAPPED (v8)':   (1 + x) ** 3 / (1 - x),
}
r_pole = {
    'single (u=1)': Rs,
    'double (u=1,2)': Rs,
    'iso mis-mapped': r[np.argmin(np.abs(L - 88.0))],
    'iso MAPPED (v8)': Rs
}

def trace(b, r_arr, n_arr, r_cap, ds=0.15):
    ln = np.log(np.minimum(n_arr, 1e12))
    X, Y, vx, vy = -150.0, float(b), 1.0, 0.0  # <-- LÍNEA 46 CORREGIDA

    def w(xx, yy):
        rr = max(np.hypot(xx, yy), 1e-3)
        d = 0.05
        lp = np.interp(rr + d, r_arr, ln)
        lm = np.interp(max(rr - d, 1e-3), r_arr, ln)
        g = np.clip((lp - lm) / (2 * d), -50, 50)
        return g * xx / rr, g * yy / rr

    def f(xx, yy, vxx, vyy):
        wx, wy = w(xx, yy)
        vd = vxx * wx + vyy * wy
        return wx - vd * vxx, wy - vd * vyy

    for _ in range(20000):
        rr = np.hypot(X, Y)
        if rr <= r_cap + 0.2:
            return True
        if X > 150.0 or rr > 250.0:
            return False

        # Etapas RK4 acopladas (posición + velocidad)
        v1x, v1y = vx, vy
        k1x, k1y = f(X, Y, v1x, v1y)

        v2x, v2y = vx + 0.5 * ds * k1x, vy + 0.5 * ds * k1y
        k2x, k2y = f(X + 0.5 * ds * v1x, Y + 0.5 * ds * v1y, v2x, v2y)

        v3x, v3y = vx + 0.5 * ds * k2x, vy + 0.5 * ds * k2y
        k3x, k3y = f(X + 0.5 * ds * v2x, Y + 0.5 * ds * v2y, v3x, v3y)

        v4x, v4y = vx + ds * k3x, vy + ds * k3y
        k4x, k4y = f(X + ds * v3x, Y + ds * v3y, v4x, v4y)

        X += (ds / 6.0) * (v1x + 2 * v2x + 2 * v3x + v4x)
        Y += (ds / 6.0) * (v1y + 2 * v2y + 2 * v3y + v4y)
        vx += (ds / 6.0) * (k1x + 2 * k2x + 2 * k3x + k4x)
        vy += (ds / 6.0) * (k1y + 2 * k2y + 2 * k3y + k4y)

        n_norm = np.hypot(vx, vy)
        vx, vy = vx / n_norm, vy / n_norm

    return True

def find_bc(r_arr, n_arr, r_cap, b_min=10.0, b_max=220.0, tol=0.05):
    lo, hi = b_min, b_max
    while (hi - lo) > tol:
        mid = 0.5 * (lo + hi)
        if trace(mid, r_arr, n_arr, r_cap):
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)

# --- Ejecución del fork calibrado ---
print(f"Horizonte exterior Rs = {Rs:.1f}\n")
for name, n_arr in COMPLETIONS.items():
    rc = r_pole[name]
    bc = find_bc(r, n_arr, rc)
    print(f"[{name:17s}] r_cap={rc:5.1f}  b_c={bc:6.2f}  k = b_c/Rs = {bc/Rs:.3f}")

# Control analítico GR isotrópico (2*GM = Rs)
GM = 0.5 * Rs
r_gr = np.linspace(GM / 2.0 + 1e-4, 220.0, 2000)
n_gr = (1.0 + GM / (2.0 * r_gr)) ** 3 / (1.0 - GM / (2.0 * r_gr))
bc_gr = find_bc(r_gr, n_gr, GM / 2.0)
print(f"\n[GR-control (RK4) ] r_cap={GM/2.0:5.1f}  b_c={bc_gr:6.2f}  k_cal = {bc_gr/Rs:.3f} (teórico: 2.598)")


Horizonte exterior Rs = 59.0

[single (u=1)     ] r_cap= 59.0  b_c=174.02  k = b_c/Rs = 2.950
[double (u=1,2)   ] r_cap= 59.0  b_c=164.74  k = b_c/Rs = 2.792
[iso mis-mapped   ] r_cap= 12.0  b_c= 21.63  k = b_c/Rs = 0.367
[iso MAPPED (v8)  ] r_cap= 59.0  b_c=169.64  k = b_c/Rs = 2.875

[GR-control (RK4) ] r_cap= 14.8  b_c=112.53  k_cal = 1.907 (teórico: 2.598)
